# QC-Py-Cloud-03 — Dual Momentum : Asset Selection Matters

**Module**: Hands-On AI Trading, Ch.6 — Momentum Strategies

**Objectif**: Implementer et comparer des variantes du Dual Momentum (Antonacci, 2014) sur QuantConnect Cloud, en demontrant l'impact critique de la selection d'actifs defensifs sur les performances.

**Approche cloud-native**: L'algorithme est execute sur QuantConnect Cloud. Les resultats sont syncs ci-dessous.

## 1. Concept : Dual Momentum

Le Dual Momentum combine deux signaux :

1. **Momentum absolu** (time-series) : Un actif a un rendement positif sur une periode donnee (ex: 12 mois). Si oui, il est "en tendance". Si non, on se refugie en defensif.

2. **Momentum relatif** (cross-section) : Parmi les actifs en tendance, on selectionne les meilleurs performers.

**Reference** : Gary Antonacci, "Dual Momentum Investing" (2014).

### Pourquoi c'est pedagogiquement riche

Le Dual Momentum semble simple en theorie, mais sa performance depend massivement de **la selection des actifs defensifs**. La "bonne" reponse (BND, TLT, SHY, cash) change selon le regime de taux d'interet. C'est un excellent cas d'etude pour comprendre que l'edge d'une strategie reside souvent dans les details d'implementation, pas dans le signal lui-meme.

## 2. Les trois variantes

### v1 — Original (Antonacci classique)

| Parametre | Valeur |
|-----------|--------|
| Actifs risques | SPY, EFA |
| Actif defensif | BND (Vanguard Total Bond) |
| Selection | Top 1 par momentum 12m |
| Allocation | 100% dans l'actif choisi |
| Filtre absolu | 12m return > 0 |
| Rebalance | Mensuel |

### v2 — NoTLT (diversified defensive)

| Parametre | Valeur |
|-----------|--------|
| Actifs risques | SPY, QQQ, IEF, GLD, XLP |
| Actif defensif | Cash (liquidate) |
| Selection | Top 2 par momentum 12m |
| Allocation | Equal-weight (50% chacun) |
| Filtre absolu | 12m return > 0 |
| Rebalance | Mensuel |

**Changement cle** : L'univers inclut deja des actifs defensifs (IEF, GLD, XLP) qui peuvent etre selectionnes par le momentum relatif. Pas besoin d'un safe asset separé.

### v3 — Momentum-Weighted (allocation proportionnelle)

| Parametre | Valeur |
|-----------|--------|
| Actifs risques | SPY, QQQ, EFA, GLD |
| Actif defensif | SHY (Treasury 1-3y) |
| Selection | Top 3 par momentum 12m |
| Allocation | Momentum-weighted (proportionnel au score) |
| Filtre absolu | 12m return > 0 |
| Rebalance | Mensuel |

## 3. Resultats QC Cloud

**Projet QC Cloud**: 30822524 (`Cloud-DualMomentum-NoTLT`)
**Periode**: 2014-01-01 au 2025-01-01 (11 ans)
**Capital initial**: $100,000

| Version | Univers | Allocation | Sharpe | CAGR | MaxDD | Net Profit | Ordres | Win Rate |
|---------|---------|------------|--------|------|-------|------------|--------|----------|
| v1 (Original) | SPY, EFA + BND | 100% single | 0.340 | 8.08% | 33.6% | +135.1% | 260 | 67% |
| **v2 (NoTLT)** | **SPY, QQQ, IEF, GLD, XLP** | **Equal top 2** | **0.392** | **8.79%** | **23.6%** | **+152.8%** | **250** | **78%** |
| v3 (Weighted) | SPY, QQQ, EFA, GLD + SHY | Mom-weighted top 3 | 0.322 | 7.62% | 22.8% | +124.4% | 369 | 79% |

### Benchmark : SPY Buy & Hold
Sur la meme periode, SPY a un CAGR ~12.8% et un MaxDD ~24%.

## 4. Analyse : L'impact de l'actif defensif

### v1 : Le piege BND (et TLT)

La v1 utilise BND comme refuge defensif. En theorie, les obligations protegent quand les actions chutent. En pratique :

- **2020 (COVID)** : BND a fonctionne comme prevu (+2% quand SPY -34%)
- **2022 (rate hikes)** : BND a perdu ~13%. SPY perdait aussi ~25%. Les deux ont chute ensemble.

Le MaxDD de 33.6% provient principalement de 2022, quand BND ne jouait plus son role de refuge.

### v2 : La diversification integree

La v2 elimine le besoin d'un actif defensif separe en incluant deja des actifs defensifs dans l'univers :

- **GLD** : or, refuge en crise et inflation
- **IEF** : obligations 7-10 ans, moins sensibles aux taux que TLT
- **XLP** : biens de consommation de base, defensif en recession

Quand SPY et QQQ ont un momentum negatif, le filtre absolu les elimine. Le portefeuille se tourne naturellement vers GLD, IEF ou XLP, qui ont souvent un momentum positif quand les actions chutent.

**Resultat** : MaxDD reduit de 33.6% a 23.6%, Sharpe ameliore de 0.340 a 0.392.

### v3 : Le piege du momentum-weighting

La v3 pondererait par le score de momentum. Le probleme est identique a Cloud-02 : la concentration sur les actifs a momentum eleve augmente le risque de retournement.

Avec 4 actifs et top 3 selectionnes, le portefeuille est souvent fortement concentre sur 1-2 actifs (les poids sont proportionnels aux scores, pas egaux). Le Sharpe (0.322) est inferieur a l'equal-weight v2.

## 5. Lecon principale : L'edge est dans l'univers, pas le signal

Les trois versions utilisent le meme signal (momentum 12 mois) et le meme filtre absolu (return > 0). La seule difference est l'univers d'actifs.

| Changement | Impact Sharpe | Impact MaxDD |
|------------|---------------|-------------|
| BND -> diversifie | +0.052 | -10.0% |
| Equal-weight -> momentum-weighted | -0.070 | +0.8% (negligeable) |

**Conclusion** : La selection de l'univers (quels actifs, quelles classes d'actifs) a un impact plus important que la methode d'allocation (equal-weight vs momentum-weighted).

C'est coherent avec les recherches academiques : la diversification asset-class explique la majorite de la variance de performance dans les strategies multi-actifs.

## 6. Code source (v2 — meilleur resultat)

Le code est disponible sur QC Cloud (projet 30822524). Le notebook local ne contient que les resultats et l'analyse, conformement au workflow cloud-native.

```python
# Lien QC Cloud : https://www.quantconnect.com/project/30822524
# Approche : Dual Momentum NoTLT, top 2 equal-weight
# Univers : SPY, QQQ, IEF, GLD, XLP
# Filtre : 12m absolute momentum > 0
# Rebalance mensuel, pas d'actif defensif separe
#
# Points cles :
# 1. Les actifs defensifs (IEF, GLD, XLP) font partie de l'univers
# 2. Le filtre absolu elimine automatiquement les actifs en bear market
# 3. Top 2 equal-weight = diversification suffisante sans sur-complexite
```

## 7. Comparaison avec les autres cloud-native QuantBooks

| Strategie | Sharpe | CAGR | MaxDD | Net Profit |
|-----------|--------|------|-------|------------|
| Risk Parity v4 (Cloud-01) | 0.278 | 6.17% | 20.4% | +93.3% |
| **Sector Rotation v3 (Cloud-02)** | **0.614** | **10.76%** | **15.5%** | **+207.8%** |
| **Dual Momentum v2 (Cloud-03)** | **0.392** | **8.79%** | **23.6%** | **+152.8%** |

Le Sector Rotation v3 (trend-following multi-asset) reste le meilleur. Le Dual Momentum v2 offre un bon compromis rendement/risque avec une approche differente (momentum classique sans filtre tendance).

## 8. Pour aller plus loin

1. **Lookback adaptatif** : Tester un lookback variable (6m en bear, 12m en bull) pour s'adapter au regime.

2. **Volatility scaling** : Reduire l'exposition quand la volatilite implcite (VIX) est elevee.

3. **Safe asset dynamique** : Choisir entre SHY, IEF, GLD selon le regime de taux (taux montants = SHY, taux stables = IEF, inflation = GLD).

4. **Acceleration signal** : Combiner momentum 12m avec acceleration du momentum (pente du momentum) pour detecter les retournements plus tot.

**Reference** : Antonacci, G. (2014) — "Dual Momentum Investing: An Innovative Strategy for Higher Returns with Lower Risk", McGraw-Hill.

In [1]:
# Cellule Cloud-only : le code est execute sur QC Cloud, pas localement
# Voir les resultats dans la section 3 ci-dessus
print("Notebook QC-Py-Cloud-03 — Resultats sync depuis QC Cloud projet 30822524")

Notebook QC-Py-Cloud-03 — Resultats sync depuis QC Cloud projet 30822524
